In [ ]:
"""
- Income
- Loan Amount
- Monthly / Weekly
- Payment History - Ontime/ Advanced / Late
-
"""

In [5]:
### Libraries
from trino.dbapi import connect
import pandas as pd
conn = connect(
    host = "202.4.114.56",
    port = "8888",
    user = "gazu",
    catalog= "padakhep",
    schema= "gold",
)
cur = conn.cursor()

def query_to_df(sql):
    cur.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

In [11]:
### total inactive_loan
inactive_loan=query_to_df('''select * from loans where  is_active = 'No' ''')
inactive_loan.shape

(1694190, 22)

In [24]:
inactive_loan_member=query_to_df('''select  count(distinct member_id) from loans where  is_active = 'No' ''')
inactive_loan_member

,_col0
0,853464


In [17]:
### Not Active but have different loan_condition
query_to_df('''select loan_condition from loans where  is_active = 'No' ''').value_counts()

loan_condition
regular           1694156
watchful               33
sub_standard            1
Name: count, dtype: int64

In [30]:
## Check tenure end loan with condition
lc = query_to_df(''' select member_id, loan_condition  from loans where next_unpaid_schedule_id = 0 ''')
lc['loan_condition'].value_counts()

loan_condition
regular         1661136
bad_loan         140432
doubtful          16308
sub_standard       6580
watchful            238
Name: count, dtype: int64

In [32]:
## Member count of each loan_condition group
lc.groupby('loan_condition').agg(
    loan_count = ('loan_condition', 'size'),
    member_count = ('member_id', 'nunique'),
)

,loan_count,member_count
loan_condition,,
bad_loan,140432,135011
doubtful,16308,15888
regular,1661136,846199
sub_standard,6580,6542
watchful,238,238


In [37]:
bad_loan_member= lc[lc['loan_condition']=='bad_loan'].groupby('member_id').size()
(bad_loan_member >= 2).sum()

np.int64(5274)

In [40]:
members_more_than_2 = (
    lc[lc['loan_condition'] == 'bad_loan']
    .groupby('member_id')
    .size()
    .loc[lambda x: x > 2]
    .sort_values(ascending=False)
)

members_more_than_2

member_id
1071981    7
1072786    7
1070631    6
511430     4
568156     4
          ..
862106     3
1013257    3
984732     3
862236     3
1071821    3
Length: 133, dtype: int64